In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/lung_cancer.csv")

# Tidy the column names: strip stray spaces, lower-case, use underscores
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print("Shape:", df.shape)
print(df.columns.tolist())
df.head()

Shape: (309, 16)
['gender', 'age', 'smoking', 'yellow_fingers', 'anxiety', 'peer_pressure', 'chronic_disease', 'fatigue', 'allergy', 'wheezing', 'alcohol_consuming', 'coughing', 'shortness_of_breath', 'swallowing_difficulty', 'chest_pain', 'lung_cancer']


,gender,age,smoking,yellow_fingers,anxiety,peer_pressure,chronic_disease,fatigue,allergy,wheezing,alcohol_consuming,coughing,shortness_of_breath,swallowing_difficulty,chest_pain,lung_cancer
0,M,69,1,2,2,1,1,2,1,2,2,2,2,2,2,YES
1,M,74,2,1,1,1,2,2,2,1,1,1,2,2,2,YES
2,F,59,1,1,1,2,1,2,1,2,1,2,2,1,2,NO
3,M,63,2,2,2,1,1,1,1,1,2,1,1,2,2,NO
4,F,63,1,2,1,1,1,1,1,2,1,2,2,1,1,NO


In [2]:
print("Data types:")
print(df.dtypes)

print("\nMissing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
print(df["lung_cancer"].value_counts())
print((df["lung_cancer"].value_counts(normalize=True) * 100).round(1))

print("\nValues in each column:")
for col in df.columns:
    if col != "age":
        print(f"{col}: {sorted(df[col].unique())}")

Data types:
gender                     str
age                      int64
smoking                  int64
yellow_fingers           int64
anxiety                  int64
peer_pressure            int64
chronic_disease          int64
fatigue                  int64
allergy                  int64
wheezing                 int64
alcohol_consuming        int64
coughing                 int64
shortness_of_breath      int64
swallowing_difficulty    int64
chest_pain               int64
lung_cancer                str
dtype: object

Missing values: 0
Duplicate rows: 33

Target distribution:
lung_cancer
YES    270
NO      39
Name: count, dtype: int64
lung_cancer
YES    87.4
NO     12.6
Name: proportion, dtype: float64

Values in each column:
gender: ['F', 'M']
smoking: [np.int64(1), np.int64(2)]
yellow_fingers: [np.int64(1), np.int64(2)]
anxiety: [np.int64(1), np.int64(2)]
peer_pressure: [np.int64(1), np.int64(2)]
chronic_disease: [np.int64(1), np.int64(2)]
fatigue: [np.int64(1), np.int64(2)]
allergy: 

In [3]:
df_clean = df.copy()

# 1. Remove exact duplicate rows
before = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f"Rows: {before} -> {len(df_clean)} ({before - len(df_clean)} duplicates removed)")

# 2. Recode gender: 1 = male, 0 = female (same convention as the other modules)
df_clean["male"] = (df_clean["gender"] == "M").astype(int)
df_clean = df_clean.drop(columns="gender")

# 3. Recode the survey's 1/2 answers (1 = NO, 2 = YES) to 0/1
binary_cols = [c for c in df_clean.columns if c not in ["age", "male", "lung_cancer"]]
for c in binary_cols:
    df_clean[c] = df_clean[c] - 1

# 4. Recode the target: YES -> 1, NO -> 0
df_clean["lung_cancer"] = df_clean["lung_cancer"].map({"YES": 1, "NO": 0})

# Fixed, documented column order: predictors first, target last
feature_order = ["male", "age"] + binary_cols
df_clean = df_clean[feature_order + ["lung_cancer"]]

print("\nFeature order (we will copy this into Flask later):")
print(feature_order)

print("\nTarget distribution after cleaning:")
print(df_clean["lung_cancer"].value_counts())
print((df_clean["lung_cancer"].value_counts(normalize=True) * 100).round(1))

print("\nSanity check, unique values in the first three recoded columns:")
print({c: sorted(df_clean[c].unique().tolist()) for c in binary_cols[:3]})

df_clean.to_csv("../data/processed/lung_cancer_clean.csv", index=False)
print("\nSaved to ml/data/processed/lung_cancer_clean.csv")
df_clean.head()

Rows: 309 -> 276 (33 duplicates removed)

Feature order (we will copy this into Flask later):
['male', 'age', 'smoking', 'yellow_fingers', 'anxiety', 'peer_pressure', 'chronic_disease', 'fatigue', 'allergy', 'wheezing', 'alcohol_consuming', 'coughing', 'shortness_of_breath', 'swallowing_difficulty', 'chest_pain']

Target distribution after cleaning:
lung_cancer
1    238
0     38
Name: count, dtype: int64
lung_cancer
1    86.2
0    13.8
Name: proportion, dtype: float64

Sanity check, unique values in the first three recoded columns:
{'smoking': [0, 1], 'yellow_fingers': [0, 1], 'anxiety': [0, 1]}

Saved to ml/data/processed/lung_cancer_clean.csv


,male,age,smoking,yellow_fingers,anxiety,peer_pressure,chronic_disease,fatigue,allergy,wheezing,alcohol_consuming,coughing,shortness_of_breath,swallowing_difficulty,chest_pain,lung_cancer
0,1,69,0,1,1,0,0,1,0,1,1,1,1,1,1,1
1,1,74,1,0,0,0,1,1,1,0,0,0,1,1,1,1
2,0,59,0,0,0,1,0,1,0,1,0,1,1,0,1,0
3,1,63,1,1,1,0,0,0,0,0,1,0,0,1,1,0
4,0,63,0,1,0,0,0,0,0,1,0,1,1,0,0,0


In [5]:
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

print("Rows after cleaning:", len(df_clean))
print("Class counts:", df_clean["lung_cancer"].value_counts().to_dict())

X = df_clean.drop(columns="lung_cancer")
y = df_clean["lung_cancer"]

# 80:20 stratified split, same seed as the other modules. The test set stays untouched for now.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "| negatives:", (y_train == 0).sum())
print("Test: ", X_test.shape, "| negatives:", (y_test == 0).sum())

models = {
    "Baseline (always YES)": DummyClassifier(strategy="most_frequent"),
    "Random Forest": RandomForestClassifier(random_state=42, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Support Vector Machine": SVC(random_state=42, class_weight="balanced"),
}

cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=5, random_state=42)
scoring = {
    "accuracy": "accuracy",
    "balanced_acc": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

rows = []
for name, model in models.items():
    pipe = make_pipeline(StandardScaler(), model)
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    rows.append({"Model": name, **{k: res[f"test_{k}"].mean() for k in scoring}})

results_df = pd.DataFrame(rows).sort_values("balanced_acc", ascending=False).round(3)
results_df

Rows after cleaning: 276
Class counts: {1: 238, 0: 38}
Train: (220, 15) | negatives: 30
Test:  (56, 15) | negatives: 8


,Model,accuracy,balanced_acc,precision,recall,f1,roc_auc
4,Support Vector Machine,0.862,0.788,0.949,0.889,0.916,0.898
1,Random Forest,0.884,0.767,0.938,0.927,0.931,0.896
3,K-Nearest Neighbors,0.885,0.692,0.915,0.957,0.934,0.849
2,Decision Tree,0.847,0.684,0.915,0.908,0.910,0.686
0,Baseline (always YES),0.864,0.500,0.864,1.000,0.927,0.500


In [6]:
from sklearn.metrics import (confusion_matrix, recall_score, precision_score,
                             f1_score, roc_auc_score, balanced_accuracy_score)

# SVM with probability output (needed for the risk levels), scaler inside the pipeline
svm_pipe = make_pipeline(StandardScaler(),
                         SVC(random_state=42, class_weight="balanced", probability=True))
svm_pipe.fit(X_train, y_train)

pred = svm_pipe.predict(X_test)
proba = svm_pipe.predict_proba(X_test)[:, 1]

tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
print(f"Test set: {len(y_test)} patients ({tp+fn} cancer, {tn+fp} no cancer)")
print(f"Cancers found (TP): {tp} | Cancers missed (FN): {fn}")
print(f"Healthy cleared (TN): {tn} | False alarms (FP): {fp}")
print()
print(f"Sensitivity (recall): {recall_score(y_test, pred):.3f}")
print(f"Specificity:          {tn/(tn+fp):.3f}")
print(f"Precision:            {precision_score(y_test, pred):.3f}")
print(f"F1-score:             {f1_score(y_test, pred):.3f}")
print(f"Balanced accuracy:    {balanced_accuracy_score(y_test, pred):.3f}")
print(f"ROC-AUC:              {roc_auc_score(y_test, proba):.3f}")
print(f"(Always-YES baseline accuracy on this test set: {(y_test == 1).mean():.3f})")

# How are the predicted probabilities spread? This decides how we define the risk levels.
print("\nPredicted probability of lung cancer, by true outcome:")
print(pd.Series(proba[y_test.values == 1]).describe().round(2).rename("true cancer"))
print(pd.Series(proba[y_test.values == 0]).describe().round(2).rename("true no cancer"))

Test set: 56 patients (48 cancer, 8 no cancer)
Cancers found (TP): 44 | Cancers missed (FN): 4
Healthy cleared (TN): 7 | False alarms (FP): 1

Sensitivity (recall): 0.917
Specificity:          0.875
Precision:            0.978
F1-score:             0.946
Balanced accuracy:    0.896
ROC-AUC:              0.964
(Always-YES baseline accuracy on this test set: 0.857)

Predicted probability of lung cancer, by true outcome:
count    48.00
mean      0.93
std       0.14
min       0.31
25%       0.96
50%       0.97
75%       0.98
max       0.99
Name: true cancer, dtype: float64
count    8.00
mean     0.44
std      0.22
min      0.19
25%      0.31
50%      0.39
75%      0.49
max      0.86
Name: true no cancer, dtype: float64


In [7]:
import joblib

scaler = svm_pipe.named_steps["standardscaler"]
svc = svm_pipe.named_steps["svc"]

joblib.dump(svc, "../saved_models/lung_cancer_model.pkl")
joblib.dump(scaler, "../saved_models/lung_cancer_scaler.pkl")
print("Saved lung_cancer_model.pkl and lung_cancer_scaler.pkl")

# Reload the saved pair and prove it reproduces the results, using the same two steps Flask uses
m = joblib.load("../saved_models/lung_cancer_model.pkl")
s = joblib.load("../saved_models/lung_cancer_scaler.pkl")
proba_reloaded = m.predict_proba(s.transform(X_test.values))[:, 1]
print("Max difference vs original probabilities:", float(np.abs(proba_reloaded - proba).max()))

# Verified demo cases from the test set
check = pd.DataFrame({"true": y_test.values, "pred": pred, "prob": proba.round(3)}, index=X_test.index)
pos_idx = check[(check.true == 1) & (check.pred == 1)].sort_values("prob").index[-1]
neg_idx = check[(check.true == 0) & (check.pred == 0)].sort_values("prob").index[0]

print("\n=== VERIFIED CANCER CASE (correct), probability:", check.loc[pos_idx, "prob"])
print(X_test.loc[pos_idx].to_dict())
print("\n=== VERIFIED NO-CANCER CASE (correct), probability:", check.loc[neg_idx, "prob"])
print(X_test.loc[neg_idx].to_dict())

print("\nFeature order for Flask:")
print(X.columns.tolist())

Saved lung_cancer_model.pkl and lung_cancer_scaler.pkl
Max difference vs original probabilities: 0.0

=== VERIFIED CANCER CASE (correct), probability: 0.993
{'male': 0, 'age': 65, 'smoking': 0, 'yellow_fingers': 1, 'anxiety': 1, 'peer_pressure': 1, 'chronic_disease': 1, 'fatigue': 0, 'allergy': 1, 'wheezing': 1, 'alcohol_consuming': 1, 'coughing': 1, 'shortness_of_breath': 1, 'swallowing_difficulty': 1, 'chest_pain': 0}

=== VERIFIED NO-CANCER CASE (correct), probability: 0.186
{'male': 0, 'age': 57, 'smoking': 1, 'yellow_fingers': 1, 'anxiety': 0, 'peer_pressure': 0, 'chronic_disease': 0, 'fatigue': 0, 'allergy': 0, 'wheezing': 0, 'alcohol_consuming': 0, 'coughing': 0, 'shortness_of_breath': 1, 'swallowing_difficulty': 0, 'chest_pain': 0}

Feature order for Flask:
['male', 'age', 'smoking', 'yellow_fingers', 'anxiety', 'peer_pressure', 'chronic_disease', 'fatigue', 'allergy', 'wheezing', 'alcohol_consuming', 'coughing', 'shortness_of_breath', 'swallowing_difficulty', 'chest_pain']
